In [1]:
import main
import pandas as pd
from analysis import load_results_experiment
from analysis.analyze_results import generate_reports_for_paths, StrPath
from arc_cegis import experiment



In [2]:
# import glob
# from typing import cast

# PATHS: list[StrPath] = cast(list[StrPath], glob.glob("experiments/**/results_experiment_100_100.json", recursive=True))
filename = "results_experiment_100_100"
def model_name_to_path(provider, name):
    return f"experiments/{provider}/{name}/{filename}.json"

PATHS: dict[tuple[str, str], StrPath] = {
    ("google",       "gemini-3.1-flash-lite",):    "Gemini 3.1 Flash Lite",
    ("google",       "gemini-3.5-flash-lite",):    "Gemini 3.5 Flash Lite",
    ("pollinations", "chigwell/claude-sonnet-5",): "Claude Sonnet 5" ,
    ("pollinations", "chigwell/grok-4.6",):        "Grok 4.6",
    ("pollinations", "morriszdweck/glm-fast",):    "GLM 5.3 Flash",
    ("pollinations", "openai",):                   "GPT-5.4 Nano",
    ("pollinations", "gpt-5.6-luna",):             "GPT-5.6 Luna",
    ("pollinations", "deepseek",):                 "DeepSeek V4 Flash 0731",
    ("mistral",      "labs-leanstral-1-5-1",):     "Leanstral 1.5.1" ,
}

In [3]:
import shutil
from pathlib import Path

def clear_directory(dir_path):
    folder = Path(dir_path)
    
    # Iterate through all items inside the directory
    for item in folder.iterdir():
        try:
            if item.is_file() or item.is_symlink():
                item.unlink()  # Delete files or symbolic links
            elif item.is_dir():
                shutil.rmtree(item)  # Delete subdirectories and their contents
        except Exception as e:
            print(f"Failed to delete {item}. Reason: {e}")

In [4]:
model_paths: list[StrPath]=[model_name_to_path(provider, model) for provider, model in PATHS.keys()]

prefix = "experiments/reports"

generate_reports_for_paths(model_paths, prefix, strategy="cegis")

from pathlib import Path

generate_reports_for_paths(model_paths, prefix, strategy="cegis_anticheat")


for (provider, model), friendlyname in PATHS.items():
    for cegis in ("cegis", "cegis_anticheat"):
        parent = Path("reports", friendlyname, cegis)
        parent.mkdir(parents=True, exist_ok=True)
        clear_directory(parent)
        Path(prefix, model, cegis, filename).move(
            parent
        )


In [5]:
import analysis.generate_experiment_summary as ges


for (provider, model), friendlyname in PATHS.items():
    results = Path(model_name_to_path(provider, model))
    markdown = Path("summary", f"{friendlyname}.md")
    ges.summarize_file(results, markdown)

In [6]:
import analysis.generate_latex_table as glt

paths, labels = zip(*(
    (Path(model_name_to_path(provider, model)), friendlyname)
    for (provider, model), friendlyname
    in PATHS.items()
))
paths=list(paths)
labels=list(labels)

with open("docs/summary_table.tex", "wt") as f:
    f.write(glt.generate_combined_latex_table(paths, labels))